In [2]:
import networkx as nx
import pandas as pd
import torch
import pickle

## 1. extract protein embedddings

In [3]:
def load_graph(filepath):
    with open(filepath, 'rb') as f:
        G = pickle.load(f)
    print(f"Load graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    return G

In [8]:
G = load_graph("../../EdgeAssignmentTask/neighborhood_selection/results/pykeen_kge/AD_KG/G_merge_AD_KG_healthy.pkl")

protein_nodes = []
for node, attr in G.nodes(data=True):
    # print(node)
    # print(attr)

    if attr.get('label','') == 'Protein':
        protein_nodes.append(node)
    # break

Load graph with 7820 nodes and 24461 edges


In [7]:
len(protein_nodes)

2245

In [10]:
df_embedding = pd.read_csv("../../EdgeAssignmentTask/neighborhood_selection/results/pykeen_kge/AD_KG/embedding.csv", index_col=0)
embeddings = df_embedding[df_embedding.index.isin(protein_nodes)]
embeddings

,Component_1,Component_2,Component_3,Component_4,Component_5,Component_6,Component_7,Component_8,Component_9,Component_10,...,Component_247,Component_248,Component_249,Component_250,Component_251,Component_252,Component_253,Component_254,Component_255,Component_256
"p(CHEBI:""amyloid-beta polypeptide 42"")",-0.838868,-0.894812,-1.665828,-0.544204,-3.117030,-2.072838,-0.127108,0.739820,1.961052,-2.418546,...,0.842275,-0.236417,-0.191154,-1.368578,-0.375749,0.474667,0.663514,-0.414643,-0.038772,-2.395431
"p(CHEBI:""amyloid-beta"")",0.296918,0.303208,0.231687,0.017844,0.004659,0.132376,-0.039097,0.163802,-0.232891,0.196605,...,0.578156,0.033174,0.184513,0.191104,0.201498,0.162784,-0.037819,0.310565,-0.474039,0.208725
"p(CHEBI:""cholesterol"")",-1.612742,-1.458753,2.437611,-0.055724,-1.314280,0.290219,0.310820,0.897895,0.628886,0.065758,...,-1.217489,-0.778933,-0.942460,-0.172261,-0.661084,-1.984987,-0.728508,0.095277,1.586002,-0.674741
"p(CONSO:""APP C-terminally truncated carboxyl-terminal fragments"")",0.010255,0.306495,-3.339482,-0.005943,-0.400550,-0.502296,-1.220509,0.500731,0.047272,1.457177,...,-0.134347,1.284306,0.072084,-0.284287,0.555389,0.390514,-1.781293,1.313754,-0.156425,0.027209
"p(CONSO:""CDK5R1 p25"")",0.119553,0.422912,-0.126679,0.481594,0.787591,0.343737,0.159694,-0.588059,-0.143606,-0.220296,...,0.181886,0.617813,0.031614,0.645528,-0.047753,-0.096054,0.039567,-0.367092,0.357344,-0.564393
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"p(UniProtKB:""VLDLR_RAT"")",-0.238215,0.217475,-1.015950,-4.427256,0.293364,-0.042365,-0.475032,-0.266107,0.393299,1.241113,...,0.573863,1.430837,-0.259398,1.168401,-2.742837,1.496125,0.770096,-1.276596,-0.742884,1.000416
"p(UniProtKB:""VLDLR_RAT"",var(""O-linked glycosylation domain_del""))",-0.439215,-1.923677,1.165449,-0.110652,1.207234,-0.202278,2.042466,-0.775431,0.611722,-1.361381,...,-0.086732,0.599778,0.689020,0.902289,0.840229,0.755602,-0.089410,0.982870,0.314392,0.261414
"p(UniProtKB:""VPS35_HUMAN"")",-0.009121,0.515759,-0.728741,-0.062857,-0.058965,0.455048,-0.412247,0.849345,0.153839,-0.200507,...,0.371100,0.922370,-0.060566,-0.226681,-0.437978,-0.003002,-0.485556,0.295658,-0.718291,-0.033799
"p(UniProtKB:""VTDB_MOUSE"")",0.210360,-0.446954,0.293490,-1.524487,-1.331538,0.407359,0.320301,-0.081390,-0.842339,-1.081459,...,-1.134385,0.121495,-1.411129,-1.432221,0.475971,-1.593193,0.561471,-0.214770,-0.365978,0.175966


## 2. create patient-protein network
+ patient features: gene expression
+ proteins features: embeddings
+ edge_weight: normalized expression value
+ edge_type: with/without specifying disease/control

In [ ]:
class PatientNetworkGenerator:
    def __init__(self):
        """
        Initializes with knowledge graphs. All input graphs are forced to MultiDiGraph.
        """
        
        self.relation_map = {1: 'up_reg', -1: 'down_reg'}
        self.pattern_hgnc =  r'^p\(HGNC:"([^"]+)"\)$'
        self.pattern_uniprotkb =  r'^p\(UniProtKB:"([^"_%]+)_[A-Z]+"\)$'
        self.pattern_dash = r'(\w+)_HUMAN'

    def gene_symbol_extractor(self, text, pattern:str):
        # ^ ensures start at the beginning, $ ensures end at the ')'
        match = re.search(pattern, text)
        if match:
            target = match.group(1)
            return target.upper()
        return None

    def get_symbol_mapping(self, graph):
        """
        Universal helper function to create {gene_symbol: [kg_nodes]} mapping 
        across ad_kg, control_kg, ppi_kg, and prime_kg.
        """
        mapping = defaultdict(list)
        if graph is None: 
            return dict(mapping)
        
        for node, attr in graph.nodes(data=True):
            # Ensure node is a string and skip empty/complex modifications
            if not isinstance(node, str) or any(x in node for x in ['frag(', 'var(', 'pmod(', 'loc(']):
                continue
                
            symbol = None
            
            # Case 1: BEL Namespace syntax (ad_kg & control_kg) -> p(HGNC:"...") or p(UniProtKB:"...")
            if node.startswith('p(') and node.endswith(')'):
                if 'HGNC' in node:
                    symbol = self.gene_symbol_extractor(node, self.pattern_hgnc)
                elif 'UniProtKB' in node:
                    symbol = self.gene_symbol_extractor(node, self.pattern_uniprotkb)
                else:
                    # Fallback for unexpected namespaces inside p(...) like p(FIXME:"...")
                    match = re.search(r'["\']([^"\']+)["\']', node)
                    if match:
                        symbol = match.group(1)
                    #continue

            # Case 2: handle (ppi_kg) -> AL1A1_HUMAN
            elif '_' in node and any(suffix in node for suffix in ['_HUMAN']):
                if 'UniProtKB' not in node and 'HGNC' not in node:
                    symbol = self.gene_symbol_extractor(node, self.pattern_dash)

            else:
                # Case 3: handle Gene/Protein symbols in PrimeKg
                clean_node = node.strip().upper()
                if attr.get('label') and attr.get('label') in 'Gene/Protein':
                    symbol = clean_node

            # Append to our list mapping if a valid symbol was resolved
            if symbol:
                symbol_upper = symbol.upper()
                mapping[symbol_upper].append(node)
                
        return dict(mapping)
    
    def generate(self, 
                 data: pd.DataFrame, 
                 exp_df: pd.DataFrame, 
                 protein_embeddings:pd.DataFrame) -> Tuple[nx.MultiGraph, pd.DataFrame]:
        """
        Connects samples to KG protein nodes.
        """
        patient_labels = data['label'].to_dict()
        scores = data.drop(columns=['label'])
        
        
        #self.relation_map = {1: f'up_reg', -1: f'down_reg'}

        overlay_graph = nx.MultiDiGraph()
        
        symbol_to_kg_node = self.get_symbol_mapping(overlay_graph)
        common_proteins = [s for s in scores.columns if s in symbol_to_kg_node]
        
        sparse_data = scores[common_proteins].stack()
        radicals = sparse_data[sparse_data != 0]

        summary_df = pd.DataFrame(0, index=data.index, columns=['pos_edges', 'neg_edges'])

        summary_df['linked_nodes'] = [[] for _ in range(len(summary_df))] # Initialize with empty lists
        summary_df['label'] = data['label'].to_list()

        # add patient nodes to overlay graph
        overlay_graph = add_patient_attrs(G=overlay_graph,
                                          features_df=exp_df,
                                          labels_series=data['label'],
                                          )
        # add edges between patient and protein
        for key, val in tqdm(radicals.items(), total=len(radicals), desc="Linking Samples"):
            if not (isinstance(key, tuple) and len(key) == 2):
            # unexpected index shape — skip or handle explicitly
                continue
            patient, symbol = key

            for protein_node in symbol_to_kg_node[symbol]:
                rel = self.relation_map.get(int(val))
                
                if not overlay_graph.has_node(patient):
                    overlay_graph.add_node(patient, label=patient_labels[patient], type='Patient')

                weight_value = float(exp_df.loc[patient, symbol])
                
                # Use relation as the 'key' for MultiDiGraph edges
                overlay_graph.add_edge(patient, protein_node, key=rel, relation=rel, weight=weight_value, label=patient_labels[patient])
                overlay_graph.add_edge(protein_node, patient, key=f'rev_{rel}', relation=f'rev_{rel}',weight=weight_value,label=patient_labels[patient])
                
                col = 'pos_edges' if int(val) == 1 else 'neg_edges'
                summary_df.at[patient, col] += 1
                summary_df.at[patient,'linked_nodes'].append(protein_node)

        return overlay_graph, summary_df
    

In [15]:
from collections import defaultdict
import re


pattern_hgnc =  r'^p\(HGNC:"([^"]+)"\)$'
pattern_uniprotkb =  r'^p\(UniProtKB:"([^"_%]+)_[A-Z]+"\)$'
pattern_dash = r'(\w+)_HUMAN'

def gene_symbol_extractor(text, pattern:str):
    # ^ ensures start at the beginning, $ ensures end at the ')'
    match = re.search(pattern, text)
    if match:
        target = match.group(1)
        return target.upper()
    return None

def get_symbol_mapping(nodes:list):
    """
    Universal helper function to create {gene_symbol: [kg_nodes]} mapping 
    across ad_kg, control_kg, ppi_kg, and prime_kg.
    """
    mapping = defaultdict(list)
    
    for node in nodes:
        # Ensure node is a string and skip empty/complex modifications
        if not isinstance(node, str) or any(x in node for x in ['frag(', 'var(', 'pmod(', 'loc(']):
            continue
            
        symbol = None
        
        # Case 1: BEL Namespace syntax (ad_kg & control_kg) -> p(HGNC:"...") or p(UniProtKB:"...")
        if node.startswith('p(') and node.endswith(')'):
            if 'HGNC' in node:
                symbol = gene_symbol_extractor(node, pattern_hgnc)
            elif 'UniProtKB' in node:
                symbol = gene_symbol_extractor(node,pattern_uniprotkb)
            else:
                # Fallback for unexpected namespaces inside p(...) like p(FIXME:"...")
                match = re.search(r'["\']([^"\']+)["\']', node)
                if match:
                    symbol = match.group(1)
                #continue

        # Case 2: handle (ppi_kg) -> AL1A1_HUMAN
        elif '_' in node and any(suffix in node for suffix in ['_HUMAN']):
            if 'UniProtKB' not in node and 'HGNC' not in node:
                symbol = gene_symbol_extractor(node, pattern_dash)

        else:
            # Case 3: handle Gene/Protein symbols in PrimeKg
            clean_node = node.strip().upper()

        # Append to our list mapping if a valid symbol was resolved
        if symbol:
            symbol_upper = symbol.upper()
            mapping[symbol_upper].append(node)
            
    return dict(mapping)

In [16]:
symbol_node_map = get_symbol_mapping(list(embeddings.index))
symbol_node_map

{'AMYLOID-BETA POLYPEPTIDE 42': ['p(CHEBI:"amyloid-beta polypeptide 42")'],
 'AMYLOID-BETA': ['p(CHEBI:"amyloid-beta")'],
 'CHOLESTEROL': ['p(CHEBI:"cholesterol")'],
 'APP C-TERMINALLY TRUNCATED CARBOXYL-TERMINAL FRAGMENTS': ['p(CONSO:"APP C-terminally truncated carboxyl-terminal fragments")'],
 'CDK5R1 P25': ['p(CONSO:"CDK5R1 p25")'],
 'N-TERMINAL FRAGMENT OF APP': ['p(CONSO:"N-terminal fragment of APP")'],
 'YENPTY ENDOCYTOSIS MOTIF (APP)': ['p(CONSO:"YENPTY endocytosis motif (APP)")'],
 'SAPP-ALPHA': ['p(CONSO:"sAPP-alpha")'],
 'SAPP-BETA': ['p(CONSO:"sAPP-beta")'],
 '1.11.1.9': ['p(ECCODE:"1.11.1.9")'],
 '1.15.1.1': ['p(ECCODE:"1.15.1.1")'],
 '1.9.3.1': ['p(ECCODE:"1.9.3.1")'],
 'A4_751_RAT': ['p(FIXME:"A4_751_RAT")'],
 'APP751/770': ['p(FIXME:"APP751/770")'],
 'AMYLOID BETA-PROTEIN (40-1)': ['p(FIXME:"amyloid beta-protein (40-1)")'],
 'IMMATURE APP695': ['p(FIXME:"immature APP695")'],
 'LAMBDA PROTEIN PHOSPHATASE': ['p(FIXME:"lambda protein phosphatase")'],
 'MISFOLDED SOD1': ['p(